# Diciplina de Machine Learning - Pós-Graduação PUC-RIO
### MVP apresentado como atividade final da Sprint de Machine Learning

Criação de modelo de Machine Learning destinado à classificação de risco de quadro de hipertensão.

Objetivo: Aplicação em dispositivos de monitoramento físico e aplicativos de treinamento físico.

Conjunto de dados empregado para análise: Hypertension Risk Prediction Dataset

Este conjunto de dados sintéticos, porém realistas, foi criado para auxiliar pesquisadores, cientistas de dados e entusiastas da saúde a analisar os fatores de risco associados à hipertensão (pressão alta). Ele contém 1.985 registros e 11 recursos significativos gerados com base em insights clínicos e padrões de dados de saúde pública.

| Variável | Descrição |
| :---: | --- |
| Age | Idade do paciente (em anos) |
| Salt_Intake | Ingestão diária de sal (em gramas) – um dos principais fatores que contribuem para a pressão alta |
| Stress_Score | Escala de 0 a 10 que mede o nível de estresse psicológico |
| BP_History | Estado pressórico anterior: Normal, Pré-hipertensão, Hipertensão |
| Sleep_Duration | Média de horas de sono por dia |
| BMI | Índice de Massa Corporal (medida de obesidade baseada em peso/altura) |
| Medication | Tipo de medicação: Nenhum, Betabloqueador, Diurético, Inibidor da ECA, Outro |
| Family_History | Histórico familiar de hipertensão: Sim/Não |
| Exercise_Level | Nível de atividade física: Baixo, Moderado, Alto |
| Smoking_Status | Se o paciente é fumante ou não fumante |
| Has_Hypertension | Variável-alvo: Indica a presença de hipertensão (Sim/Não) |

Fonte: https://www.kaggle.com/datasets/miadul/hypertension-risk-prediction-dataset

<a href="https://colab.research.google.com/github/rogerioferreira/MVP-Machine_Learnin-PUC-RIO/blob/main/notebooks/hypertension_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importação das bibliotecas necessárias para o projeto

In [ ]:
# importação de bibliotecas básicas
import os
import joblib
import warnings
import pandas as pd
import plotly.express as px

# importação de métodos de tratamento
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA

# importação de modelos
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# importação de ensembles
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.pipeline import Pipeline

# configuração para não exibir os warnings
warnings.filterwarnings("ignore")

# definição da semente aleatória visando a reprodutibilidade
SEED = 42

## Obtenção, análise e tratamento dos dados

In [80]:
data_path = "../dados/hypertension_dataset.csv"
url = "https://raw.githubusercontent.com/rogerioferreira/MVP-Machine_Learning-PUC-RIO/refs/heads/main/dados/hypertension_dataset.csv"

if os.path.exists(data_path):
    # caraga do dataset localmente, caso presente
    df = pd.read_csv(data_path, delimiter=",")
else:
    # caraga do dataset diretamente pelo Git Hub
    df = pd.read_csv(url, delimiter=",")

### Verificação da ocorrência de dados nulos no dataframe

Foi oservada a presença de valores nulos no registro do uso de medicação continuada para uma parcela significativa das ocorrências (40% dos casos), contudo, conforme descrição constante nos próprios metadados no Kaggle, o tipo de medicação subdivide-se em Nenhum, Betabloqueador, Diurético, Inibidor da ECA ou Outro, contudo, percebeu-se que os registros nulos referiam-se à aussência de medicação, desta forma, o procedimento adotado foi a substituição dos campos nulos com a atribuição do valor "No Medication", de forma a viabilizar de maneira adequada o emprego da informação no treinamento do modelo.

In [81]:
# verificação da presença de valores nulos
df.isnull().sum()

Age                   0
Salt_Intake           0
Stress_Score          0
BP_History            0
Sleep_Duration        0
BMI                   0
Medication          799
Family_History        0
Exercise_Level        0
Smoking_Status        0
Has_Hypertension      0
dtype: int64

In [82]:
# calculo do percentual de dados nulos
percentual = (df["Medication"].isnull().sum() / len(df)) * 100
print(f"Percentual de dados nulos: {percentual:.2f}%")

Percentual de dados nulos: 40.25%


In [83]:
# observação quanto a ausência de categorigazação nos casos de nenhuma medicação
df["Medication"].unique().tolist()

[nan, 'ACE Inhibitor', 'Other', 'Beta Blocker', 'Diuretic']

In [84]:
# atribuição de informação de não uso de medicação nos registros nulos
df["Medication"].fillna("No Medication", inplace=True)

### Verificação de dados duplicados no dataframe

Não foram observadas ocorrências de duplicidades nos dados.

In [85]:
# verificação da presença de dados duplicados
print(f"Identificação de duplicatas: {df.duplicated().any()}")

# verificação da quantidade de entradas duplicadas no dataframe
print(f"Quantidade de registro duplicados: {df.duplicated().sum()}")

Identificação de duplicatas: False
Quantidade de registro duplicados: 0


### Verificação da existência de outliers

Foram analisadas as variáveis Idade, Ingestão Diária de Sal, Escala de Estresse Psicológico, Horas de Sono e Índice de Massa Corporal quanto a ocorrência de outliers, por tratarem-se de variáveis não categóricas, chegando-se às seguintes conclusões:

As variáveis Idade e Escala de Estresse Psicológico não apresentaram nenhuma ocorrência fora dos limites dos limites máximos e mínimos com base nos quartis, afastando automaticamente a hipótese da existência de outliers para estes casos.

As variáveis Ingestão Diária de Sal e Horas de Sono, apesar de apresentarem valores inferiores e superiores aos limites mínimo e máximo do primeiro e do terceiro quartis, respectivamente, observa-se que os mesmos encontram-se muito próximos às áreas limítrofes e com valores coerentes às variáveis, não havendo dispersão significativa, optando-se, desta forma, pelo afastamento da hipótese da caracterização dos mesmo como outliers.

Por fim, na análise da variável Índice de Massa Corporal, observa-se uma dispersão um pouco maior dos valores inferiores e superiores aos limites mínimo e máximo do primeiro e do terceiro quartis, respectivamente, contudo, entende-se que os mesmos são valores acitáveis para ocorrência de índice de massa corporal (cisnes negros), devendo os mesmos serem empregados como valores válidos ao invés de serem descartados, optando-se, desta forma, pelo afastamento da hipótese da caracterização dos mesmo como outliers.

In [86]:
# identificação de outliers nas variáveis não categóricas ou binárias
features = ["Age", "Salt_Intake", "Stress_Score", "Sleep_Duration", "BMI"]


# boxplot para visualização de outliers
fig = px.box(
    df[features],
    y=features,
    title="Identificação de outliers",
    width=800,
    height=500,
)
fig.update_layout(xaxis_title="Variaveis", yaxis_title="Valores")
fig.show()

In [87]:
# exibição dos valores mínimos e máximos das variáveis analisadas
features = ["Salt_Intake", "Sleep_Duration", "BMI"]

for feature in features:
    print(
        f"{feature:<14} - Min: {df[feature].min():>5.2f}   Max: {df[feature].max():>5.2f}"
    )

Salt_Intake    - Min:  2.50   Max: 16.40
Sleep_Duration - Min:  1.50   Max: 11.40
BMI            - Min: 11.90   Max: 41.90


In [88]:
df[features].describe()

,Salt_Intake,Sleep_Duration,BMI
count,1985.000000,1985.000000,1985.000000
mean,8.531688,6.452242,26.015315
std,1.994907,1.542207,4.512857
min,2.500000,1.500000,11.900000
25%,7.200000,5.400000,23.000000
50%,8.500000,6.500000,25.900000
75%,9.900000,7.500000,29.100000
max,16.400000,11.400000,41.900000


In [89]:
df.head()

,Age,Salt_Intake,Stress_Score,BP_History,Sleep_Duration,BMI,Medication,Family_History,Exercise_Level,Smoking_Status,Has_Hypertension
0,69,8.0,9,Normal,6.4,25.8,No Medication,Yes,Low,Non-Smoker,Yes
1,32,11.7,10,Normal,5.4,23.4,No Medication,No,Low,Non-Smoker,No
2,78,9.5,3,Normal,7.1,18.7,No Medication,No,Moderate,Non-Smoker,No
3,38,10.0,10,Hypertension,4.2,22.1,ACE Inhibitor,No,Low,Non-Smoker,Yes
4,41,9.8,1,Prehypertension,5.8,16.2,Other,No,Moderate,Non-Smoker,No


### Tratamento de variáveis categóricas e variáveis numéricas

Como técnica de tratamento das variáveis Estado Pressórico Anterior e Tipo de Medicação, por tratarem-se de variáveis categóricas não binárias, optou-se pelo emprego do método "One Hot Encoder", criando-se uma coluna binária para cada ocorrência de tipo categórico na variável. Além disso, ouptou-se pela remoção da primeira ocorrência na criação das novas variáveis binárias, de forma a evitar a multicolinearidade nos dados resultantes.

Já para as variáveis categóricas binárias Histórico Familiar de Hipertensão, Fumante ou não Fumante e Presença de Hipertensão (variável-alvo), optou-se pela adoção do método "Ordinal Encoder" em virtude da característica binária das mesmas.

Para a variável Nível de Atividade Física, apesar da mesma não se tratar de uma variável categórica binária, optou-se pelo emprego do método "Ordinal Encoder", ordenando-se suas categorias em Baixo, Moderado e Alto, de forma a dar sentido de peso, atribuindo-se para as mesmas os valores 0, 1 e 2, respctivamente.

Por fim, as variáveis numéricas Idade e Escala de Estresse Psicológico foram transformadas de valores inteiros de 64 bits para valores de ponto flutuante de 64 bits.

In [90]:
# realização da transformação de variáveis categóricas não binárias
# drop="first" - evita a multicolinearidade, removendo a primeira categoria
features = ["BP_History", "Medication"]
one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")

one_hot_features = one_hot.fit_transform(df[features])
df_one_hot = pd.DataFrame(
    one_hot_features,
    columns=one_hot.get_feature_names_out(features),
)

df.drop(columns=features, inplace=True)  # remoção das features categóricas
df = pd.concat([df_one_hot, df], axis=1)  # concatenação dos dados transformados

del df_one_hot  # exclusão do dataframe temporário

In [91]:
# realização da transformação de variáveis categóricas binárias ou com sentido sequencial
features = []

features.append(("Family_History", ["No", "Yes"]))
features.append(("Smoking_Status", ["Non-Smoker", "Smoker"]))
features.append(("Has_Hypertension", ["No", "Yes"]))
features.append(("Exercise_Level", ["Low", "Moderate", "High"]))


for feature, ordem in features:
    one_hot = OrdinalEncoder(handle_unknown="error", categories=[ordem])

    one_hot_feature = one_hot.fit_transform(df[[feature]])
    df_one_hot = pd.DataFrame(
        one_hot_feature,
        columns=[feature],
    )

    df.drop(columns=feature, inplace=True)  # remoção das features categóricas
    df = pd.concat([df_one_hot, df], axis=1)  # concatenação dos dados transformados

    del df_one_hot  # exclusão do dataframe temporário

In [92]:
# conversão das features com valores inteiros para ponto flutuante
df[["Age", "Stress_Score"]] = df[["Age", "Stress_Score"]].astype("float64")

In [93]:
# verificação da estrutura final do dataset após tratamento das variaveis
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1985 entries, 0 to 1984
Data columns (total 15 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Exercise_Level              1985 non-null   float64
 1   Has_Hypertension            1985 non-null   float64
 2   Smoking_Status              1985 non-null   float64
 3   Family_History              1985 non-null   float64
 4   BP_History_Normal           1985 non-null   float64
 5   BP_History_Prehypertension  1985 non-null   float64
 6   Medication_Beta Blocker     1985 non-null   float64
 7   Medication_Diuretic         1985 non-null   float64
 8   Medication_No Medication    1985 non-null   float64
 9   Medication_Other            1985 non-null   float64
 10  Age                         1985 non-null   float64
 11  Salt_Intake                 1985 non-null   float64
 12  Stress_Score                1985 non-null   float64
 13  Sleep_Duration              1985 

### Verificação de correlação entre as variáveis independentes

In [94]:
# verificação da correlação entre as variáveis
df.drop(columns="Has_Hypertension").corr()

# necessidade de implementação de gráfico

,Exercise_Level,Smoking_Status,Family_History,BP_History_Normal,BP_History_Prehypertension,Medication_Beta Blocker,Medication_Diuretic,Medication_No Medication,Medication_Other,Age,Salt_Intake,Stress_Score,Sleep_Duration,BMI
Exercise_Level,1.000000,0.005716,0.012351,0.006072,-0.008149,-0.001422,-0.015379,0.035178,-0.005235,0.016352,0.014880,0.000988,-0.021225,0.002327
Smoking_Status,0.005716,1.000000,-0.010821,0.000518,-0.008203,0.027781,-0.030858,0.003112,-0.003458,0.003457,-0.029285,0.029000,0.039781,0.004546
Family_History,0.012351,-0.010821,1.000000,0.016462,-0.014474,0.003867,0.016620,-0.021534,0.012483,-0.000619,-0.049213,-0.008549,-0.012063,0.018093
BP_History_Normal,0.006072,0.000518,0.016462,1.000000,-0.551436,0.001989,-0.019522,0.011727,-0.005418,0.004099,0.005863,0.048243,-0.031990,-0.012460
BP_History_Prehypertension,-0.008149,-0.008203,-0.014474,-0.551436,1.000000,0.006205,-0.018255,-0.007895,0.005908,0.016249,-0.012943,-0.002835,-0.000909,-0.001372
Medication_Beta Blocker,-0.001422,0.027781,0.003867,0.001989,0.006205,1.000000,-0.192040,-0.420064,-0.174153,0.029237,-0.018969,-0.019964,-0.006706,0.003219
Medication_Diuretic,-0.015379,-0.030858,0.016620,-0.019522,-0.018255,-0.192040,1.000000,-0.307992,-0.127689,-0.013676,0.007938,-0.033123,-0.005860,0.007858
Medication_No Medication,0.035178,0.003112,-0.021534,0.011727,-0.007895,-0.420064,-0.307992,1.000000,-0.279304,-0.013821,0.031256,0.036461,0.003837,0.011740
Medication_Other,-0.005235,-0.003458,0.012483,-0.005418,0.005908,-0.174153,-0.127689,-0.279304,1.000000,0.016720,-0.004992,-0.006176,0.025861,-0.017777
Age,0.016352,0.003457,-0.000619,0.004099,0.016249,0.029237,-0.013676,-0.013821,0.016720,1.000000,0.011205,-0.032184,-0.011839,-0.023498


### Preparação dos dados para seleção de modelos

In [95]:
# Separação em bases de treino e teste (holdout)
X = df.drop(columns="Has_Hypertension")
y = df["Has_Hypertension"]

### Aplicação de técnica de redução de dimencionalidade

Foi empregado o método PCA como técnica de redução de dimensionalidade, resultando na consolidação de 13 para 8 caracterŕisticas relevantes para o treinaento, presenvando-se 95% da variância.

In [96]:
# aplicação de ténica de redução de dimensionalidade
pca = PCA(
    n_components=9, random_state=SEED
)  # n_components=0.97 mantém 97% da variância
X = pca.fit_transform(X)

X.shape

(1985, 9)

In [97]:
# separação dos conjuntos de treinamento e de teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)  # faz a divisão

# definição de parâmetros para a validação cruzada
num_particoes = 10  # número de folds da validação cruzada (3, 5, 10)
scoring = "accuracy"

kfold = KFold(
    n_splits=num_particoes, shuffle=True, random_state=SEED
)  # faz o particionamento em K folds

## Modelagem e inferência

### Avaliação prévia dos modelos

In [98]:
# criação de lista para armazenar os modelos
models = []

# definição de parâmetros do classificador
num_trees = 100
max_features = X.shape[1]  # número de features após PCA

# definição dos modelos para teste
models.append(("LR", LogisticRegression(random_state=SEED, max_iter=200)))
models.append(("KNN", KNeighborsClassifier()))
models.append(("DT", DecisionTreeClassifier(random_state=SEED)))
models.append(("GNB", GaussianNB()))
models.append(("SVM", SVC(random_state=SEED)))


models.append(("ADA", AdaBoostClassifier(random_state=SEED, n_estimators=num_trees)))
models.append(
    (
        "RF",
        RandomForestClassifier(
            random_state=SEED, n_estimators=num_trees, max_features=max_features
        ),
    )
)
models.append(
    (
        "ET",
        ExtraTreesClassifier(
            random_state=SEED, n_estimators=num_trees, max_features=max_features
        ),
    )
)
models.append(
    (
        "BAG",
        BaggingClassifier(
            random_state=SEED, n_estimators=num_trees, max_features=max_features
        ),
    )
)
models.append(
    (
        "GB",
        GradientBoostingClassifier(
            random_state=SEED, n_estimators=num_trees, max_features=max_features
        ),
    )
)


# criação de listas para armazenar os resultados e os nomes dos modelos
results = []
names = []

# criação de listas para armazenar as médias e os desvios padrão para geração de dataframe
means = []
std = []


# métricas mais relevantes para seleção dos modelos:
# Acurácia (accuracy): A proporção de predições corretas sobre o total de predições.
# Precisão (precision): Dos casos que o modelo previu como positivos, quantos eram realmente positivos.
# Sensibilidade (recall): Dos casos que eram realmente positivos, quantos o modelo conseguiu identificar.
# Pontuação F1 (f1-score): É a média harmônica entre precisão e recall. É uma métrica útil quando as classes estão desbalanceadas.
# Curva ROC e AUC: Medem a capacidade do modelo de distinguir entre as classes.
for name, model in models:

    print(" " * 80, end="\r")
    print(f"Treinando o modelo: {name}", end="\r")
    cv_results = cross_val_score(model, X_train, y_train, cv=kfold, scoring=scoring)

    results.append(cv_results)
    names.append(name)
    means.append(cv_results.mean())
    std.append(cv_results.std())


# geraão do dataframe de resultados
df_results = pd.DataFrame(
    {
        "Modelo": names,
        "Média": means,
        "Desvio Padrão": std,
    }
)

# exibição dos resultados
print(df_results.set_index("Modelo").sort_values("Média", ascending=False))

           Média  Desvio Padrão                                                 
Modelo                         
ADA     0.919425       0.020162
GB      0.903666       0.020052
BAG     0.879759       0.025266
ET      0.879118       0.018056
RF      0.877864       0.020189
DT      0.821825       0.034680
GNB     0.761309       0.023937
LR      0.684504       0.023171
SVM     0.663705       0.035264
KNN     0.642314       0.040503


In [99]:
# boxplot de comparação dos modelos
fig = px.box(
    data_frame=pd.DataFrame(
        data=results, index=names
    ).transpose(),  # resultados individuais em linhas (index)
    y=names,
    title=f"Comparação da Acurácia dos Modelos - método {scoring}",
    width=1000,
    height=500,
)
fig.update_layout(xaxis_title="Modelos", yaxis_title=scoring)
fig.show()

### Avaliação dos modelos com dados originais, padronizados, normalizados e robustos

In [100]:
# criação de listas para armazenar os pipelines e os resultados
pipelines = []
results = []
names = []

# criação de listas para armazenar as médias e os desvios padrão para geração de dataframe
means = []
std = []

# definição dos elementos do pipeline

# algoritmos empregados
lr = ("LR", LogisticRegression(random_state=SEED, max_iter=200))
knn = ("KNN", KNeighborsClassifier())
dt = ("DT", DecisionTreeClassifier(random_state=SEED))
gnb = ("GNB", GaussianNB())
svm = ("SVM", SVC(random_state=SEED))


ada = ("ADA", AdaBoostClassifier(random_state=SEED, n_estimators=num_trees))
rf = (
    "RF",
    RandomForestClassifier(
        random_state=SEED, n_estimators=num_trees, max_features=max_features
    ),
)
et = (
    "ET",
    ExtraTreesClassifier(
        random_state=SEED, n_estimators=num_trees, max_features=max_features
    ),
)
bag = (
    "BAG",
    BaggingClassifier(
        random_state=SEED, n_estimators=num_trees, max_features=max_features
    ),
)
gb = (
    "GB",
    GradientBoostingClassifier(
        random_state=SEED, n_estimators=num_trees, max_features=max_features
    ),
)

# transformações empregadas
standard_scaler = ("StandardScaler", StandardScaler())
min_max_scaler = ("MinMaxScaler", MinMaxScaler())
robust_scaler = ("RobustScaler", RobustScaler())

# definição dos pipelines
pipelines.append(("LR-orig", Pipeline(steps=[lr])))
pipelines.append(("LR-std", Pipeline(steps=[standard_scaler, lr])))
pipelines.append(("LR-norm", Pipeline(steps=[min_max_scaler, lr])))
pipelines.append(("LR-rob", Pipeline(steps=[robust_scaler, lr])))

pipelines.append(("KNN-orig", Pipeline(steps=[knn])))
pipelines.append(("KNN-std", Pipeline(steps=[standard_scaler, knn])))
pipelines.append(("KNN-norm", Pipeline(steps=[min_max_scaler, knn])))
pipelines.append(("KNN-rob", Pipeline(steps=[robust_scaler, knn])))

pipelines.append(("DT-orig", Pipeline(steps=[dt])))
pipelines.append(("DT-std", Pipeline(steps=[standard_scaler, dt])))
pipelines.append(("DT-norm", Pipeline(steps=[min_max_scaler, dt])))
pipelines.append(("DT-rob", Pipeline(steps=[robust_scaler, dt])))

pipelines.append(("GNB-orig", Pipeline(steps=[gnb])))
pipelines.append(("GNB-std", Pipeline(steps=[standard_scaler, gnb])))
pipelines.append(("GNB-norm", Pipeline(steps=[min_max_scaler, gnb])))
pipelines.append(("GNB-rob", Pipeline(steps=[robust_scaler, gnb])))

pipelines.append(("SVM-orig", Pipeline(steps=[svm])))
pipelines.append(("SVM-std", Pipeline(steps=[standard_scaler, svm])))
pipelines.append(("SVM-norm", Pipeline(steps=[min_max_scaler, svm])))
pipelines.append(("SVM-rob", Pipeline(steps=[robust_scaler, svm])))

pipelines.append(("ADA-orig", Pipeline(steps=[ada])))
pipelines.append(("ADA-std", Pipeline(steps=[standard_scaler, ada])))
pipelines.append(("ADA-norm", Pipeline(steps=[min_max_scaler, ada])))
pipelines.append(("ADA-rob", Pipeline(steps=[robust_scaler, ada])))

pipelines.append(("RF-orig", Pipeline(steps=[rf])))
pipelines.append(("RF-std", Pipeline(steps=[standard_scaler, rf])))
pipelines.append(("RF-norm", Pipeline(steps=[min_max_scaler, rf])))
pipelines.append(("RF-rob", Pipeline(steps=[robust_scaler, rf])))

pipelines.append(("ET-orig", Pipeline(steps=[et])))
pipelines.append(("ET-std", Pipeline(steps=[standard_scaler, et])))
pipelines.append(("ET-norm", Pipeline(steps=[min_max_scaler, et])))
pipelines.append(("ET-rob", Pipeline(steps=[robust_scaler, et])))

pipelines.append(("BAG-orig", Pipeline(steps=[bag])))
pipelines.append(("BAG-std", Pipeline(steps=[standard_scaler, bag])))
pipelines.append(("BAG-norm", Pipeline(steps=[min_max_scaler, bag])))
pipelines.append(("BAG-rob", Pipeline(steps=[robust_scaler, bag])))

pipelines.append(("GB-orig", Pipeline(steps=[gb])))
pipelines.append(("GB-std", Pipeline(steps=[standard_scaler, gb])))
pipelines.append(("GB-norm", Pipeline(steps=[min_max_scaler, gb])))
pipelines.append(("GB-rob", Pipeline(steps=[robust_scaler, gb])))


# avaliação dos pipelines
for name, model in pipelines:

    print(" " * 80, end="\r")
    print(f"Treinando o modelo: {name}", end="\r")
    cv_results = cross_val_score(model, X_train, y_train, cv=kfold, scoring=scoring)

    results.append(cv_results)
    names.append(name)
    means.append(cv_results.mean())
    std.append(cv_results.std())


# geraão do dataframe de resultados
df_results = pd.DataFrame(
    {
        "Modelo": names,
        "Média": means,
        "Desvio Padrão": std,
    }
)

# exibição dos resultados - apenas os melhores resultados
print(df_results.set_index("Modelo").sort_values("Média", ascending=False).head(12))

             Média  Desvio Padrão                                               
Modelo                           
ADA-rob   0.919425       0.020162
ADA-norm  0.919425       0.020162
ADA-std   0.919425       0.020162
ADA-orig  0.919425       0.020162
GB-orig   0.903666       0.020052
GB-std    0.903666       0.020052
GB-norm   0.903666       0.020052
GB-rob    0.903666       0.020052
BAG-rob   0.879759       0.025266
BAG-norm  0.879759       0.025266
BAG-std   0.879759       0.025266
BAG-orig  0.879759       0.025266


In [ ]:
# boxplot de comparação dos modelos
fig = px.box(
    data_frame=pd.DataFrame(
        data=results, index=names
    ).transpose(),  # resultados individuais em linhas (index)
    y=names,
    title=f"Comparação da Acurácia dos Modelos - método {scoring} - dados originais, padronizados, normalizados e robustos",
    width=1500,
    height=500,
)
fig.update_layout(xaxis_title="Modelos", yaxis_title=scoring)
fig.show()

### Otimização de hiperparâmetros

In [ ]:
# Tuning do AdaBoost
pipelines = []

# Definindo os componentes do pipeline
ada = ("ADA", AdaBoostClassifier(random_state=SEED))
standard_scaler = ("StandardScaler", StandardScaler())
min_max_scaler = ("MinMaxScaler", MinMaxScaler())
robust_scaler = ("RobustScaler", RobustScaler())

pipelines.append(("ADA-orig", Pipeline(steps=[ada])))
pipelines.append(("ADA-std", Pipeline(steps=[standard_scaler, ada])))
pipelines.append(("ADA-norm", Pipeline(steps=[min_max_scaler, ada])))
pipelines.append(("ADA-rob", Pipeline(steps=[robust_scaler, ada])))

param_grid = {
    "ADA__n_estimators": [100, 150, 200, 250],
    "ADA__learning_rate": [1.0, 1.5, 2.0, 2.5],
}

# Prepara e executa o GridSearchCV
for name, model in pipelines:
    grid = GridSearchCV(
        estimator=model, param_grid=param_grid, scoring=scoring, cv=kfold
    )
    grid.fit(X_train, y_train)

    # imprime a melhor configuração
    print(
        f"Conjunto de dados: {name} - Acurácia: {grid.best_score_:.4f} - Parâmetros: {grid.best_params_}"
    )

Conjunto de dados: ADA-orig - Acurácia: 0.9522 - Parâmetros: {'ADA__learning_rate': 1.5, 'ADA__n_estimators': 200}
Conjunto de dados: ADA-std - Acurácia: 0.9522 - Parâmetros: {'ADA__learning_rate': 1.5, 'ADA__n_estimators': 200}
Conjunto de dados: ADA-norm - Acurácia: 0.9522 - Parâmetros: {'ADA__learning_rate': 1.5, 'ADA__n_estimators': 200}
Conjunto de dados: ADA-rob - Acurácia: 0.9522 - Parâmetros: {'ADA__learning_rate': 1.5, 'ADA__n_estimators': 200}


### Avaliação do modelo com os dados de teste

In [29]:
# avaliação do modelo com o conjunto de testes

# preparação do modelo
scaler = RobustScaler().fit(X_train)
rescaledX = scaler.transform(X_train)
model = AdaBoostClassifier(random_state=SEED, n_estimators=200, learning_rate=1.5)
model.fit(rescaledX, y_train)

# estimativa da acurácia no conjunto de teste
rescaledTestX = scaler.transform(X_test)
predictions = model.predict(rescaledTestX)
print(f"Acuracia: {accuracy_score(y_test, predictions):.4f}")
print(f"Recall: {recall_score(y_test, predictions):.4f}")

Acuracia: 0.9673
Recall: 0.9610


In [ ]:
# treinamento do modelo com o dataset completo
scaler = RobustScaler().fit(X)
rescaledX = scaler.transform(X)
model.fit(rescaledX, y)

# salvando o modelo treinado para uso futuro
# joblib.dump(model, "../modelos/adaboost_robust_model.pkl")